### Installing the kaggle library (It's already present in collab but need to be installed on other platforms)

In [38]:
!pip install kaggle

### Importing the dependencies

In [39]:
# need these to access our configuration file
# using these in order to read the kaggle file
import os
import json

from zipfile import ZipFile
import pandas as pd # to load the csv file into a pandas dataframe
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

### Data collection through kaggle API

In [40]:
kaggle_dictionary = json.load(open('kaggle.json')) # json.load converts a json object into a python dictionary

In [41]:
kaggle_dictionary.keys() # the dictionary has my username and my kaggle API key

dict_keys(['username', 'key'])

In [42]:
# setup kaggle credentials as environment variables
# os.environ is a dictionary-like object where you set environment variables
# and these variables can be accessed by the current program and any libraries or child processes running within that environment.
os.environ['KAGGLE_USERNAME'] = kaggle_dictionary['username']
os.environ['KAGGLE_KEY'] = kaggle_dictionary['key']


In [43]:
# downloading the zip file from kaggle
!kaggle datasets download lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
imdb-dataset-of-50k-movie-reviews.zip: Skipping, found more recently modified local copy (use --force to force download)


In [44]:
# unzip the dataset file

with ZipFile('imdb-dataset-of-50k-movie-reviews.zip', 'r') as zip_ref: # zif_ref is a variable
  zip_ref.extractall() # extracts all the files within this compressed file

In [45]:
!ls

'IMDB Dataset.csv'			 kaggle.json
 imdb-dataset-of-50k-movie-reviews.zip	 sample_data


## Loading the dataset

In [46]:
data = pd.read_csv('IMDB Dataset.csv')

In [47]:
data.shape

(50000, 2)

In [48]:
data.head(3)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive


### Checking if we have any class imbalance

In [49]:
data['sentiment'].value_counts() # there is no class imbalance

,count
sentiment,
positive,25000
negative,25000


In [50]:
# encoding the sentiments column

# applying lambda function
data['sentiment'] = data['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)

# or it can be done this way
# data.replace({'sentiment': {'positive': 1, 'negative': 0}}, inplace=True)

# or it can also be done in this way
# data['sentiment'] = data['sentiment'].map({'positive': 1, 'negative': 0})

In [51]:
data.head(3)

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1


### Splitting the data into test and train data

In [52]:
train_data, test_data = train_test_split(data, test_size = 0.2, random_state = 42)

In [53]:
print(train_data.shape)
print(test_data.shape)

(40000, 2)
(10000, 2)


# Data preprocessing

First we will tokenize the data

tokenizer converts words to numbers

In [54]:
tokenizer = Tokenizer(num_words = 5000) # take the most common 5000 words and convert into integers

tokenizer.fit_on_texts(train_data['review']) # fit the tokenizer on the training data

X_train = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen = 200) # convert the training data into sequences of integers

X_test = pad_sequences(tokenizer.texts_to_sequences(test_data['review']), maxlen = 200) # convert the test data into sequence of integers

# so we're getting all words from the training data and assigning them indexes
# then we're writing the entire data as numbers (tokenizer.texts_to_sequences(train_data['review'])
# and then making the length of each sequence same by padding (200)
# we're not fitting on test data

In [55]:
print(X_train)

[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]


In [56]:
y_train = train_data['sentiment']
y_test = test_data['sentiment']

# Building the LSTM model

In [57]:
model = Sequential()

model.add(Input(shape=(200,)))
model.add(Embedding(input_dim = 5000, output_dim = 128)) # input_dim is the number of words in our vocabulary
model.add(LSTM(128, dropout = 0.2, recurrent_dropout = 0.2)) # 128 units, dropouts to prevent overfitting
model.add(Dense(1, activation = 'sigmoid')) # output layer, binary classification so sigmoid

In [58]:
model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 200, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 771,713 (2.94 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

In [59]:
# complie the model
model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

# Training the model

In [60]:
model.fit(X_train, y_train, validation_split = 0.2, epochs = 5, batch_size = 64)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 281s 463ms/step - accuracy: 0.7888 - loss: 0.4518 - val_accuracy: 0.8525 - val_loss: 0.3596
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 207s 414ms/step - accuracy: 0.8518 - loss: 0.3602 - val_accuracy: 0.8388 - val_loss: 0.3681
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 210s 420ms/step - accuracy: 0.8698 - loss: 0.3182 - val_accuracy: 0.8367 - val_loss: 0.3636
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 257s 409ms/step - accuracy: 0.8855 - loss: 0.2820 - val_accuracy: 0.8656 - val_loss: 0.3339
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 207s 415ms/step - accuracy: 0.9016 - loss: 0.2506 - val_accuracy: 0.8677 - val_loss: 0.3536


## Model evaluation

In [61]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"test loss = {loss}")
print(f"test accuracy = {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 37s 116ms/step - accuracy: 0.8753 - loss: 0.3353
test loss = 0.3353045880794525
test accuracy = 0.8752999901771545


## Building a predictive system

you give it a text and it predicts the sentiment

In [67]:
def predict_sentiment(review):

  # tokenize and pad the review
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence, maxlen = 200)

  # making a prediction
  prediction = model.predict(padded_sequence)

  # prediction will be a list inside a list - [[0.45, 0.55]]
  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
  return sentiment

### Example usage

In [68]:
new_review = "this movie was fantastic and i loved it"

sentiment = predict_sentiment(new_review)
print(f"the sentiment of the review is {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 430ms/step
the sentiment of the review is positive


In [69]:
new_review = "the movie was kind of good"

sentiment = predict_sentiment(new_review)
print(f"the sentiment of the review is {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
the sentiment of the review is positive


In [70]:
new_review = "this movie was not worth watching"

sentiment = predict_sentiment(new_review)
print(f"the sentiment of the review is {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step
the sentiment of the review is negative
